In [0]:
# Run this once to initialize your metadata tracking ecosystem
spark.sql("CREATE DATABASE IF NOT EXISTS petiot")

watermark_table_path = "abfss://silver@petiot.dfs.core.windows.net/meta/watermarks"

try:
    spark.read.format("delta").load(watermark_table_path)
    print("STATUS: Watermark tracking data table already initialized.")
except:
    print("STATUS: Table not found. Seeding initial baseline marks...")
    
    # Set the baseline watermark processing dates for all incremental facts
    initial_seeds = [
        ("sales", "2024-01-01 00:00:00"),
        ("device_snapshots", "2024-01-01 00:00:00"),
        ("activity_snapshots", "2024-01-01 00:00:00"),
        ("health_snapshots", "2024-01-01 00:00:00"),
        ("feeding_events", "2024-01-01 00:00:00"),
        ("hydration_events", "2024-01-01 00:00:00"),
        ("device_failures", "2024-01-01 00:00:00"),
        ("firmware_deployments", "2024-01-01 00:00:00")
    ]
    
    from pyspark.sql.types import StructType, StructField, StringType
    schema = StructType([
        StructField("table_name", StringType(), False),
        StructField("last_watermark_value", StringType(), False)
    ])
    
    df = spark.createDataFrame(initial_seeds, schema)
    df.write.format("delta").mode("overwrite").save(watermark_table_path)
    
    # Register the table in the Hive metastore so it can be queried via standard Spark SQL
    spark.sql(f"CREATE TABLE petiot.watermarks USING DELTA LOCATION '{watermark_table_path}'")
    print("STATUS: petiot.watermarks table successfully registered.")